# Voiceover Generation

Generates audio from a canonical script result. It can run against a real
content run or the committed reference fixture without invoking any LLM.
Voice behavior comes entirely from the selected profile in `workbench.yaml`.


In [ ]:
from __future__ import annotations

from copy import deepcopy
from datetime import datetime, timedelta, timezone
import json
import os
from pathlib import Path
import subprocess
import sys

import yaml

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
if not (PROJECT_ROOT / "src" / "pipeline").exists():
    raise RuntimeError("Run this notebook from the repository root or output directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pipeline.orchestration import fingerprint_token, load_run_context, resolve_script_result

OUTPUT_ROOT = PROJECT_ROOT / "output"
PIPELINE_CONFIG_PATH = PROJECT_ROOT / "src" / "pipeline" / "config" / "workbench.yaml"
CACHE_ROOT = PROJECT_ROOT / ".cache" / "ai_content_creation" / "models"
FIXTURE_DIR = PROJECT_ROOT / "tests" / "fixtures" / "media_roundup_reference"
IST = timezone(timedelta(hours=5, minutes=30))


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(path)
    value = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(value, dict):
        raise ValueError(f"Expected JSON object at {path}")
    return value


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")


def write_handoff_files(files: list[dict], output_dir: Path) -> list[str]:
    output_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for file_info in files:
        filename = file_info["filename"]
        if "/" in filename or "\\" in filename:
            raise ValueError(f"Unsafe handoff filename: {filename}")
        path = output_dir / filename
        path.write_text(file_info["content"], encoding="utf-8")
        paths.append(str(path))
    return paths


def render_config_templates(value):
    context = {
        "PROJECT_ROOT": str(PROJECT_ROOT),
        "OUTPUT_ROOT": str(OUTPUT_ROOT),
        "CACHE_ROOT": str(CACHE_ROOT),
        "RUN_ID": RUN_ID,
    }
    if isinstance(value, str):
        return value.format(**context)
    if isinstance(value, list):
        return [render_config_templates(item) for item in value]
    if isinstance(value, dict):
        return {key: render_config_templates(item) for key, item in value.items()}
    return value


In [ ]:
# INPUT_MODE: "fixture" is the deterministic smoke test; "run" uses output/runs.
INPUT_MODE = "run"
RUN_ID_OR_CURRENT = "current"
SCRIPT_SOURCE = "auto"  # "auto", "app", or "chatgpt" when INPUT_MODE="run"

workbench = yaml.safe_load(PIPELINE_CONFIG_PATH.read_text(encoding="utf-8"))
if INPUT_MODE == "fixture":
    RUN_ID = "fixture_media_roundup_reference"
    RUN_DIR = OUTPUT_ROOT / "tests" / "media_roundup_reference"
    SCRIPT_RESULT_PATH = FIXTURE_DIR / "script_generation_result.json"
    RUN_PROFILE = workbench["default_run_profile"]
    PROFILE_CONFIG = deepcopy(workbench["profiles"][RUN_PROFILE])
    script_generation_result = load_json(SCRIPT_RESULT_PATH)
else:
    context = load_run_context(output_root=OUTPUT_ROOT, run_id=RUN_ID_OR_CURRENT)
    RUN_ID = context["run_id"]
    RUN_DIR = Path(context["run_dir"])
    RUN_PROFILE = context["run_profile"]
    PROFILE_CONFIG = deepcopy(context["profile_config"])
    SCRIPT_RESULT_PATH, script_generation_result = resolve_script_result(run_dir=RUN_DIR, run_id=RUN_ID, source=SCRIPT_SOURCE)

full_script = script_generation_result["data"]["script_package"]["full_script"].strip()
if not full_script:
    raise ValueError(f"Script is empty: {SCRIPT_RESULT_PATH}")
print(f"Input mode:    {INPUT_MODE}")
print(f"Run ID:        {RUN_ID}")
print(f"Script source: {SCRIPT_RESULT_PATH}")
print(f"Script words:  {len(full_script.split())}")


## Generate Voiceover


In [ ]:
from src.pipeline.voiceover.config import resolve_style


def voiceover_variant(config: dict) -> str:
    def slug(value: object) -> str:
        text = str(value).strip().lower().replace(".", "p")
        text = "".join(ch if ch.isalnum() else "_" for ch in text)
        return "_".join(part for part in text.split("_") if part)
    return "_".join(filter(None, [slug(config.get("provider", "provider")), slug(config.get("style", "style")), slug(config.get("voice") or "default"), f"speed_{slug(config.get('speed', 'style'))}"]))

voiceover_config = render_config_templates(deepcopy(PROFILE_CONFIG.get("voiceover", {})))
if not voiceover_config:
    raise KeyError("Selected profile is missing voiceover config")
script_provenance = script_generation_result["data"]["script_package"]["provenance"]
script_source_label = "chatgpt" if script_generation_result["data"]["script_package"].get("generation_source") == "chatgpt_handoff" else "app"
variant = f"{voiceover_variant(voiceover_config)}_{script_source_label}_{fingerprint_token(script_provenance['script_fingerprint'])}"
voiceover_output_dir = RUN_DIR / "voiceover" / variant
voiceover_config["output_dir"] = str(voiceover_output_dir)

style = resolve_style(voiceover_config["style"])
chunking = voiceover_config.get("chunking") or {}
resolved_chunking = {
    "max_chars": chunking.get("chunk_max_chars") or style.chunk_max_chars,
    "max_sentences_per_chunk": chunking.get("max_sentences_per_chunk") or style.max_sentences_per_chunk,
    "chunk_pause_seconds": chunking.get("chunk_pause_seconds") if chunking.get("chunk_pause_seconds") is not None else style.chunk_pause_seconds,
    "paragraph_pause_seconds": chunking.get("paragraph_pause_seconds") if chunking.get("paragraph_pause_seconds") is not None else style.paragraph_pause_seconds,
}
script_sections = script_generation_result["data"]["script_package"].get("sections", [])
planned_beats = [beat for section in script_sections for beat in section.get("beats", [])]
if not script_sections or not planned_beats:
    raise RuntimeError("Voiceover requires canonical section-aware script beats")

voiceover_input = {
    "run": {"run_id": RUN_ID, "generated_at": datetime.now(IST).isoformat()},
    "config": voiceover_config,
    "input": {"script_generation": script_generation_result},
}
input_path = RUN_DIR / "inputs" / "voiceover" / f"{variant}.json"
write_json(input_path, voiceover_input)

voiceover_python = PROJECT_ROOT / ".venv-voiceover" / "bin" / "python"
if not voiceover_python.exists():
    raise FileNotFoundError(f"Voiceover venv Python not found: {voiceover_python}")
env = {**os.environ, "PYTHONPATH": str(PROJECT_ROOT)}
subprocess.run([str(voiceover_python), "-m", "src.pipeline.voiceover.cli", "--input-json", str(input_path)], cwd=PROJECT_ROOT, env=env, check=True)

voiceover_summary = load_json(voiceover_output_dir / "summary.json")
voiceover_package = load_json(voiceover_output_dir / "voiceover_package.json")
voiceover_result = {
    "status": "ok",
    "data": {"run_id": RUN_ID, "generated_at": voiceover_package.get("generated_at"), "voiceover_package": voiceover_package},
    "summary": voiceover_summary,
    "handoff": voiceover_package.get("handoff", {}),
}
write_json(voiceover_output_dir / "voiceover_input.json", voiceover_input)
write_json(voiceover_output_dir / "voiceover_result.json", voiceover_result)
write_json(RUN_DIR / "orchestration" / "voiceover_selection.json", {"script_result_path": str(SCRIPT_RESULT_PATH), "script_fingerprint": script_provenance["script_fingerprint"], "voiceover_package_path": str(voiceover_output_dir / "voiceover_package.json"), "voiceover_fingerprint": voiceover_package["provenance"]["voiceover_fingerprint"]})

print(f"Provider: {voiceover_summary['provider']}")
print(f"Voice:    {voiceover_summary['voice']}")
print(f"Duration: {voiceover_summary['duration_seconds']}s")
print(f"Chunks:   {voiceover_summary['chunk_count']}")
print(f"Units:    {voiceover_summary['audio_unit_count']}")
print(f"Beats:    {voiceover_summary['aligned_beat_count']}")
print(f"Audio:    {voiceover_summary['audio_path']}")


## Finalize Metadata Chapters

Uses actual section audio-unit start times. This step is deterministic and does not call an LLM.


In [ ]:
from src.pipeline.apps.metadata_finalizer import run as finalize_metadata

metadata_draft_path = RUN_DIR / "metadata_creator" / "metadata_creator_result.json"
if not metadata_draft_path.exists():
    if INPUT_MODE == "run":
        raise FileNotFoundError(f"Metadata draft not found: {metadata_draft_path}. Run content_pipeline metadata creation for the selected script first.")
    print("Metadata finalization skipped for fixture mode because no metadata draft is committed.")
else:
    metadata_creator_result = load_json(metadata_draft_path)
    metadata_finalizer_input = {
        "run": {"run_id": RUN_ID, "generated_at": datetime.now(IST).isoformat()},
        "config": deepcopy(PROFILE_CONFIG.get("metadata_finalizer", {})),
        "input": {"metadata_creator": metadata_creator_result, "script_generation": script_generation_result, "voiceover": voiceover_result},
    }
    metadata_finalizer_result = finalize_metadata(metadata_finalizer_input)
    if metadata_finalizer_result["status"] != "ok":
        raise RuntimeError(metadata_finalizer_result["summary"])
    metadata_finalizer_dir = RUN_DIR / "metadata_finalizer"
    finalized_metadata_package = metadata_finalizer_result["data"]["metadata_package"]
    write_json(metadata_finalizer_dir / "metadata_finalizer_input.json", metadata_finalizer_input)
    write_json(metadata_finalizer_dir / "metadata_finalizer_result.json", metadata_finalizer_result)
    write_json(metadata_finalizer_dir / "metadata_package.json", finalized_metadata_package)
    write_json(metadata_finalizer_dir / "youtube_metadata.json", finalized_metadata_package["youtube_metadata"])
    write_json(metadata_finalizer_dir / "summary.json", metadata_finalizer_result["summary"])
    print(f"Finalized chapters: {metadata_finalizer_result['summary']['chapter_count']}")
    print(f"Metadata output:    {metadata_finalizer_dir}")


In [ ]:
voiceover_summary
